In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('train.csv')
df.head(5)

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [6]:
#lets get unique one 
for col in df.columns:
    if df[col].dtype =='O':
        print(col,df[col].unique())

gender ['Female' 'Male' 'Other']
marital_status ['Single' 'Married' 'Divorced' 'Widowed']
education_level ['High School' "Master's" "Bachelor's" 'PhD' 'Other']
employment_status ['Self-employed' 'Employed' 'Unemployed' 'Retired' 'Student']
loan_purpose ['Other' 'Debt consolidation' 'Home' 'Education' 'Vacation' 'Car'
 'Medical' 'Business']
grade_subgrade ['C3' 'D3' 'C5' 'F1' 'D1' 'D5' 'C2' 'C1' 'F5' 'D4' 'C4' 'D2' 'E5' 'B1'
 'B2' 'F4' 'A4' 'E1' 'F2' 'B4' 'E4' 'B3' 'E3' 'B5' 'E2' 'F3' 'A5' 'A3'
 'A1' 'A2']


In [8]:
X=df.drop(['id','loan_paid_back'],axis=1)
y=df['loan_paid_back']

In [11]:
ordinal_col=['grade_subgrade','education_level']
nominal_col=['gender','marital_status','employment_status','loan_purpose']

education_order = [
    'High School',
    "Bachelor's",
    "Master's",
    'PhD',
    'Other'
]

grade_order = [
    'A1','A2','A3','A4','A5',
    'B1','B2','B3','B4','B5',
    'C1','C2','C3','C4','C5',
    'D1','D2','D3','D4','D5',
    'E1','E2','E3','E4','E5',
    'F1','F2','F3','F4','F5'
]
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import  ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
ct=ColumnTransformer(
    transformers=[('ord',OrdinalEncoder(categories=[education_order,grade_order],handle_unknown='use_encoded_value',unknown_value=-1),ordinal_col),
    ('nom',OneHotEncoder(drop="first",handle_unknown="error"),nominal_col)],
    remainder='passthrough'
)

pipeline=Pipeline(
    steps=[
        ('preprocesser',ct),
        ('classifier',XGBClassifier(
            n_estimators=100,
            learning_rate=0.1
        ))
    ]
)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
pipeline.fit(X_train,y_train)

y_pred=pipeline.predict(X_test)
prob=pipeline.predict_proba(X_test)[:,1]


In [12]:
prob

array([0.72716653, 0.99850816, 0.4773557 , ..., 0.9722367 , 0.8529391 ,
       0.8698969 ], shape=(118799,), dtype=float32)

In [14]:
from sklearn.metrics import classification_report
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.59      0.90      0.71     15779
           1       0.98      0.90      0.94    103020

    accuracy                           0.90    118799
   macro avg       0.79      0.90      0.83    118799
weighted avg       0.93      0.90      0.91    118799



In [18]:
test=pd.read_csv('test.csv')



In [30]:
sub_prob=pipeline.predict_proba(test)[:,1]
sub_prob_float = np.round(sub_prob, 1)
print(sub_prob_float)


[0.9 1.  0.5 ... 1.  1.  0.9]


In [33]:
submission = pd.DataFrame({
    'id'          : test['id'],
    'probability' : sub_prob_float
})
submission.to_csv('submission.csv', index=False)
submission.head(3)

,id,probability
0,593994,0.9
1,593995,1.0
2,593996,0.5
